In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

### 診断プロット用ヘルパー関数
以降の演習で繰り返し使います。内容を確認しておきましょう。

In [ ]:
def make_diag(measure, predict, test_y=0, test_pred=0, mode=0):
    plt.figure()
    if mode == 0:
        plt.scatter(measure, predict)
    elif mode == 'test':
        plt.scatter(measure, predict, label='train')
        plt.scatter(test_y, test_pred, label='test')
        plt.legend()
    vals = np.ravel(np.array([measure, predict]))
    lo, hi = vals.min(), vals.max()
    margin = 0.05 * (hi - lo)
    plt.plot([lo-margin, hi+margin], [lo-margin, hi+margin],
             linestyle='dashed', color='darkslategray')
    plt.xlabel('Measured', fontsize=14)
    plt.ylabel('Predicted', fontsize=14)
    plt.tight_layout()
    plt.show()

## タスク１：データの準備

目的変数を `y`（列 `'ME'`）、説明変数を `X`（`'mp-id'`, `'formula'`, `'ME'` 以外）に格納する。

In [ ]:
# 現在のディレクトリーにある 200612_Li-O-BVFFperco_rev_230515.csv ファイルを
# (Pandas/Numpy)で読み込み、オブジェクト(df/pd)に格納した。
# 上述の文章中にある()内の選択肢について正しい方を選べ

df = pd.read_csv("200612_Li-O-BVFFperco_rev_230515.csv")


In [ ]:

dftest = df # 単純に=で受け渡すと、dftestの変更はdfにも影響する
dfsave = df.copy()  # ディープコピーを作成して、元のデータフレームを変更しないようにする

# 変数と違いオブジェクトには様々なデータが格納されています。
# これらのデータを書き換えることを単純に=で受け渡すことで変更できる場合と、できない場合があります。
# 確認してみましょう。

print ("original value:",df.iloc[0,1]) # 0行1列目の値を表示
df.iloc[0,1] = 'Li3TiO3'
print ("new value:",df.iloc[0,1])

print ("df.values",df.values[0,1]) # 0行1列目の値を表示
df.values[0,1] = 'Li3ZrO3'
print ("df.values new value:",df.values[0,1])
print ("df current value:",df.iloc[0,1])

arr=df.values
print ("arr[0,1]",arr[0,1])


In [ ]:
print(dftest.iloc[0,1])
print(dfsave.iloc[0,1])

# dfを元に戻すにはどれが正解？
#df=dfsave
#df=dftest
#df=dfsave.copy()
#df=dftest.copy()

In [ ]:

# dfにサンプルが何件あるか確認し、空欄のある行を削除、
# その後dfのサンプル数が何件になっているかを確認するためには？
# 以下の命令を組み合わせてコードを書いてみてください。

# df.shape
# df.shape()
# df.lines
# df.dropna
# df.dropna()



In [ ]:

# オブジェクトdfyに目的変数MEを、オブジェクトdfxに記述子を格納するために以下の作業をしています。
# dfyに目的MEを格納するためのコードを完成させよ。
# dfXでは、drop_columnsに指定した列を削除し、欠損値のある行を削除しています。axisの操作の意味は？　また***を埋めてみましょう。

dfy = df[[***]]
dfX = df.drop(['mp-id', 'formula', 'ME_true', 'ME'], axis=***)

print('y shape:', y.shape)
print('X shape:', X.shape)

## タスク２：ランダムフォレストで回帰してみる

２－１）フィッティングと R2 の確認

In [ ]:
# 【考えてみよう②】
# - n_estimators=100 は「決定木の本数」です。
#   大きくすると精度と計算時間はそれぞれどうなりますか？        →
# - random_state=42 を設定する理由は何ですか？               →

reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X, y)
print('R2 =', reg.score(X, y))

In [ ]:
predict_y = reg.predict(X)

# 【考えてみよう③】
# - R2 が 1.0 に非常に近い値が出ました。これは「良いモデル」と言えますか？  →
# - 訓練データ全体でスコアを計算することの問題点は何でしょうか？           →

make_diag(y, predict_y)

## タスク３：過学習の確認 — Training / Test 分割

３－１）データを分割して学習する

In [ ]:
# 【考えてみよう④】
# - X.shape[0]（サンプル数）を確認して、test_size を決めてください。
# - 一般的に 0.2〜0.3 が多いですが、データが少ない場合に大きめにする理由は？ →

TEST_SIZE = 0.2  # ← この値を変えて試してみましょう

train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42)
print('訓練データ:', train_X.shape, '  テストデータ:', test_X.shape)

In [ ]:
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(train_X, train_y)

predict_train = reg.predict(train_X)
predict_test  = reg.predict(test_X)

R2 = r2_score(train_y, predict_train)
Q2 = r2_score(test_y,  predict_test)
print(f'R2（訓練）= {R2:.3f}')
print(f'Q2（テスト）= {Q2:.3f}')

# 【考えてみよう⑤】
# - R2 と Q2 の差が大きいとき、何が起きていますか？              →
# - Q2 が負になる場合はどういう意味ですか？                     →
# - R2≈1.0 かつ Q2≪1.0 のとき、ハイパーパラメータで何を調整すべき？  →

make_diag(train_y, predict_train, test_y=test_y, test_pred=predict_test, mode='test')

---
# LLMへの質問の仕方

ここからは、ChatGPT / Claude などの **LLMを使ってコードを生成** させる練習です。  
「良い質問」と「悪い質問」の違いを体験してみましょう。

## 演習：ハイパーパラメータチューニングのコードをLLMに書かせる

ランダムフォレストの代表的なハイパーパラメータ：

| パラメータ | 意味 | デフォルト |
|-----------|------|----------|
| `n_estimators` | 決定木の本数 | 100 |
| `max_depth` | 木の深さの上限 | None（無制限） |
| `max_features` | 各木で使う記述子数の上限 | `'sqrt'` |

---

### ❌ 悪い質問の例

> 「ランダムフォレストのハイパーパラメータを調整するコードを書いて」

→ 何のデータ？変数名は？評価指標は？探索範囲は？が不明なため、  
そのまま使えないコードが生成されやすい。

---

### ✅ 良い質問の構成要素

1. **目的**（何を最適化したいか）  
2. **コンテキスト**（変数名・データ形状）  
3. **制約**（探索するパラメータと範囲）  
4. **出力形式**（どんな形で結果を見たいか）

In [ ]:
# 【演習①】以下のプロンプトテンプレートの _____ を自分で埋めて、
#           ChatGPT / Claude に貼り付けてみましょう。

prompt_template = f"""
以下の変数が Python で定義済みです：
  train_X : numpy array, shape={train_X.shape}
  test_X  : numpy array, shape={test_X.shape}
  train_y : numpy array, shape={train_y.shape}
  test_y  : numpy array, shape={test_y.shape}

sklearn の RandomForestRegressor を使い、以下の条件でハイパーパラメータチューニングを行うコードを書いてください。

【調整するパラメータと探索範囲】
- n_estimators: _____  （例：50, 100, 200）
- max_depth   : _____  （例：5, 10, 20, None）
- max_features: _____  （任意・不要なら省略可）

【評価指標】
テストデータの R2スコア（Q2）を最大化したい。

【出力形式】
- 各パラメータ組み合わせの Q2 を print する
- 最後に「最良パラメータ」と「最良 Q2」を表示する
- 最良パラメータで再学習した診断プロット（散布図）も表示する
"""

print(prompt_template)

### Notebook+Ai で作成した場合

プロンプト：このセルより上のコードを読み取って、ランダムフォレストのハイパーパラメーターを調整するコードを書いてください。

In [ ]:
# from itertools import product

# # --- 探索するパラメータの候補 ---
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth'   : [5, 10, 20, None],
#     'max_features': [10, 30, 'sqrt'],
# }

# best_Q2     = -np.inf
# best_params = {}

# # --- 全組み合わせを試す ---
# keys   = list(param_grid.keys())
# values = list(param_grid.values())

# for combo in product(*values):
#     params = dict(zip(keys, combo))
#     reg_gs = RandomForestRegressor(**params, random_state=42)
#     reg_gs.fit(train_X, train_y)
#     Q2 = r2_score(test_y, reg_gs.predict(test_X))
#     print(f"n_estimators={params['n_estimators']:>3d}, "
#           f"max_depth={str(params['max_depth']):>4s}, "
#           f"max_features={str(params['max_features']):>4s}  →  Q2={Q2:.4f}")
#     if Q2 > best_Q2:
#         best_Q2     = Q2
#         best_params = params

# print(f"\n最良パラメータ: {best_params}")
# print(f"最良 Q2       : {best_Q2:.4f}")

# # --- 最良パラメータで再学習 → 診断プロット ---
# best_reg = RandomForestRegressor(**best_params, random_state=42)
# best_reg.fit(train_X, train_y)

# pred_train = best_reg.predict(train_X)
# pred_test  = best_reg.predict(test_X)

# print(f"\nR2（訓練）= {r2_score(train_y, pred_train):.3f}")
# print(f"Q2（テスト）= {r2_score(test_y, pred_test):.3f}")

# make_diag(train_y, pred_train, test_y=test_y, test_pred=pred_test, mode='test')

# # 実行後に確認しましょう：
# # Q1. コードは正常に動きましたか？エラーが出た場合、どう対処しましたか？  →
# # Q2. 最良パラメータの値は直感と一致しましたか？                     →
# # Q3. LLMへの質問文を変えると生成コードはどう変わりますか？試してみましょう →

---
# LLMコードを読む・修正する

「動かすだけ」から卒業して、**コードの意味を理解し・検証し・修正する**力を養いましょう。  

以下のコードは LLM に質問して得られた実際のコードです。  
**実行前にコードを読んで各設問に答えてから**実行してください。

## 読解タスク①：Feature Importance（重要度因子）

**実行前に** コードを読んで以下の質問に答えてください。

**Q1.** `reg.feature_importances_` が返す値の合計はいくつになりますか？  →  
**Q2.** `np.argsort(-fti)` に `-fti` とマイナスがついている理由は？  →  
**Q3.** 重要度が高い記述子を知った後、次にどんな分析をしたいですか？  →  

In [ ]:
# Feature Importance の可視化
descriptor_label = np.array(dfX.columns)
fti = reg.feature_importances_

# 上位10件の表示
sorted_idx = np.argsort(-fti)
fti_sorted = fti[sorted_idx]
descriptor_sorted = descriptor_label[sorted_idx]

print('=== 重要度上位10件 ===')
for i in range(10):
    print(f'{i+1:2d}. {descriptor_sorted[i]:<35s} {fti_sorted[i]:.4f}')

# グラフ化
step = max(1, len(descriptor_label) // 25)
plt.figure(figsize=(12, 4))
plt.bar(range(len(descriptor_label)), fti)
plt.xticks(ticks=np.arange(0, len(descriptor_label), step=step),
           labels=descriptor_label[np.arange(0, len(descriptor_label), step=step)],
           rotation=90, fontsize=8)
plt.xlabel('Descriptor')
plt.ylabel('Feature Importance')
plt.tight_layout()
plt.show()

SHAP による解釈

**Feature Importance との違い**を意識しながら読んでください。

| | Feature Importance | SHAP |
|--|--|--|
| 何を示すか | 記述子全体の平均的な重要度 | サンプルごとの寄与の方向（正/負）と大きさ |
| 過学習への感度 | やや高い | 比較的低い |


In [ ]:
import sys
!{sys.executable} -m pip install shap --user

In [ ]:
import shap

explainer = shap.TreeExplainer(reg, X)
shap_values = explainer.shap_values(X, check_additivity=False)

# plot_type を 'dot' → 'bar' に変えて違いを確認してみましょう
shap.summary_plot(shap_values, X, plot_type='dot', show=False,
                  feature_names=dfX.columns)
plt.tight_layout()
plt.show()

# 実行後に確認：
# Q4. Feature Importance と SHAP の上位記述子は一致しましたか？          →
# Q5. SHAP で「負の方向に寄与している記述子」を見つけましたか？意味は？    →

Optuna — 自動ハイパーパラメータ最適化

以下は LLM に「Optuna を使って RandomForest を最適化して」と質問して得たコードです。  
**グリッドサーチと何が違うか**を意識してください。

In [ ]:
import sys
!{sys.executable} -m pip install optuna --user

In [ ]:
# !pip install optuna  # 未インストールの場合は先頭の # を外して実行

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # 進捗ログを抑制

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_features' : trial.suggest_int('max_features', 10, X.shape[1]),
        'max_depth'    : trial.suggest_int('max_depth', 3, 50),
    }
    reg_trial = RandomForestRegressor(**params, random_state=42)
    reg_trial.fit(train_X, train_y)
    return r2_score(test_y, reg_trial.predict(test_X))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f'最良 Q2: {study.best_value:.4f}')
print(f'最良パラメータ: {study.best_trial.params}')

In [ ]:
# ── Optuna 探索過程の可視化 ──────────────────────────────────────────

import matplotlib
matplotlib.rcParams['font.family'] = 'MS Gothic'  # Windows日本語フォント

from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
    plot_slice,
)

# ① 最適化の推移
ax = plot_optimization_history(study)
ax.set_title("1. 最適化の推移（試行 vs 最良 Q2）")
ax.set_ylabel("Q2（テストR2）")
plt.tight_layout()
plt.show()

# ② ハイパーパラメータ重要度
ax = plot_param_importances(study)
ax.set_title("2. パラメータ重要度")
plt.tight_layout()
plt.show()

# ③ スライスプロット
axes = plot_slice(study, params=["n_estimators", "max_features", "max_depth"])
axes[0].get_figure().suptitle("3. 各パラメータ値と Q2 の関係")
plt.tight_layout()
plt.show()

# ④ 等高線プロット
ax = plot_contour(study, params=["n_estimators", "max_depth"])
ax.set_title("4. n_estimators x max_depth の等高線プロット")
plt.tight_layout()
plt.show()

# ⑤ 並行座標プロット
ax = plot_parallel_coordinate(study)
ax.set_title("5. 並行座標プロット（全試行）")
plt.tight_layout()
plt.show()


In [ ]:
# 最良パラメータで再学習 → 診断プロット
best_reg = RandomForestRegressor(**study.best_trial.params, random_state=42)
best_reg.fit(train_X, train_y)

pred_train = best_reg.predict(train_X)
pred_test  = best_reg.predict(test_X)

print(f'R2（訓練）= {r2_score(train_y, pred_train):.3f}')
print(f'Q2（テスト）= {r2_score(test_y, pred_test):.3f}')

make_diag(train_y, pred_train, test_y=test_y, test_pred=pred_test, mode='test')

# 実行後に確認：
# - 中盤でLLMに生成してもらったグリッドサーチと Q2 はどちらが高かったですか？ →
# - n_trials を増やすと Q2 はどう変化しますか？試してみましょう           →